# Score-Based Models, SDEs, and Probability Flow ODEs

The discrete diffusion chapter showed that image generation can be reformulated as the inversion of a long sequence of small Gaussian corruptions. The natural next question is what happens when the time step becomes very small and the number of steps becomes very large. In that regime, the noising process is better described as a stochastic differential equation, and the reverse dynamics become a continuous-time denoising flow. This perspective is not merely a mathematical curiosity. It clarifies why noise prediction, score prediction, stochastic sampling, and deterministic sampling are closely related rather than separate ideas.

This chapter therefore develops the continuous-time viewpoint in a layered way. We first recall the intuitive difference between ordinary and stochastic differential equations. We then define score functions and explain why they are the right objects to learn along a family of perturbed data distributions. After that we introduce the forward SDE, the reverse-time SDE, and the probability flow ODE. Finally, we connect these ideas back to denoising score matching and to the VP and VE families that appear in modern diffusion models.

For many students, this is the chapter where diffusion models stop looking like a clever discrete trick and start looking like a coherent probabilistic theory. The same model can now be described in at least three compatible languages: denoising regression, score estimation, and stochastic dynamics. None of these viewpoints is redundant. The denoising view makes optimization intuitive, the score view explains what function is actually being learned, and the SDE/ODE view explains how generation can proceed through either stochastic or deterministic evolution.

It is worth slowing down here because continuous-time notation can create unnecessary anxiety. The aim of this chapter is not to turn the course into a full course on stochastic calculus. It is to make clear why continuous-time diffusion models are mathematically natural and why their practical algorithms are not disconnected hacks. Even students who do not master every analytic subtlety should leave with a clear picture of the moving pieces.

## An Intuitive Digression on ODEs and SDEs

Before entering the score-based formalism, it is useful to pause for a conceptual distinction. An ordinary differential equation describes deterministic evolution. If a state $\boldsymbol{x}(t)$ follows
:::{math}
\frac{d\boldsymbol{x}}{dt} = \boldsymbol{f}(\boldsymbol{x}(t), t),
:::
then the vector field $\boldsymbol{f}$ tells us the instantaneous velocity at every point and time. Once the initial condition is fixed, the trajectory is fixed as well. Deterministic transport models, including flow matching and continuous normalizing flows, live in this world.

A stochastic differential equation adds a random perturbation at every instant. In informal notation,
:::{math}
d\boldsymbol{x}
=
\boldsymbol{f}(\boldsymbol{x}, t)\,dt
+
g(t)\,d\boldsymbol{w},
:::
where $\boldsymbol{w}$ is a Brownian motion. The first term is still a drift that pushes the state in a coherent direction. The second term injects infinitesimal Gaussian randomness. As a result, the evolution of a single trajectory is random even when the initial condition is fixed, but the distribution of trajectories follows a well-defined law. This is precisely the level at which diffusion models operate: they are less concerned with one exact path than with how probability mass evolves over time.

For students encountering this material for the first time, it helps to think of an ODE as a flow of particles in a velocity field and of an SDE as a flow of particles that are simultaneously advected and continuously shaken by noise. In image generation, the noising process is modeled by an SDE because we want the data distribution to spread gradually into a simpler reference distribution. The reverse generative process then learns how to undo this stochastic spreading.

One more intuition can help. In an ODE, nearby particles that start at the same point and follow the same vector field remain synchronized because no external randomness separates them. In an SDE, even particles with the same initial condition can diverge because Brownian noise perturbs them differently. This is exactly why an SDE is such a good language for noising: it describes not just a deterministic deformation of the data manifold, but an actual spreading of probability mass into surrounding space.

This distinction becomes practically important later when we compare stochastic reverse sampling with deterministic probability flow ODE sampling. The two procedures can share the same family of marginals even though one uses randomness and the other does not. Students often find this surprising at first, so it helps to keep the particle-flow picture in mind from the beginning.

## Scores and Perturbed Data Distributions

Let $p_t$ denote a time-dependent probability density on image space. The **score** of this density is defined by
:::{math}
\nabla_{\boldsymbol{x}} \log p_t(\boldsymbol{x}).
:::
The score points in the direction of increasing log-density, so it can be interpreted as a local vector that tells us how the density rises around the point $\boldsymbol{x}$. Unlike the density itself, the score is invariant under unknown normalization constants, which is one reason it is attractive in high-dimensional modeling.

In score-based generative modeling, the aim is not to learn only the score of the clean data distribution. Instead, one learns the scores of a whole family of perturbed distributions obtained by corrupting data with Gaussian noise of varying strength. This is crucial. The clean data distribution can be highly singular or concentrated near a thin manifold, making its score unstable or difficult to estimate directly. Once noise is added, the perturbed density becomes smoother and more regular, and the associated score becomes a better learning target.

It is helpful to interpret the score geometrically. The vector $\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})$ points toward regions where the current marginal density is higher. If a noisy sample lies in a low-density direction away from typical data, the score points back toward more likely structure. This is why score learning and denoising are so tightly linked. A good score field is, locally, a map of how probability mass should be pulled back toward realistic configurations.

One can think of a noisy face image here. If random corruption pushes one eye slightly out of alignment or inserts grain in the cheek region, the score near that corrupted point points toward configurations that look more like typical faces. The score is therefore not an abstract derivative with no operational meaning. It is a directional hint about how to nudge a sample back toward the data distribution.

There is also a normalization advantage here. Learning the density itself in high dimension is hard partly because the absolute scale of the density matters and normalizing constants are expensive. The score removes that obstacle by differentiating the log-density. One no longer needs the partition function explicitly. This is one of the main reasons score-based methods became so attractive theoretically.

## Denoising Score Matching

Suppose we observe noisy samples of the form
:::{math}
\widetilde{\boldsymbol{x}} = \boldsymbol{x} + \sigma \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}),
:::
where $\boldsymbol{x} \sim p_{gt}$. Let $p_\sigma$ denote the density of the noisy variable $\widetilde{\boldsymbol{x}}$. A neural network $\boldsymbol{s}_\theta(\widetilde{\boldsymbol{x}}, \sigma)$ can be trained to approximate the score $\nabla_{\widetilde{\boldsymbol{x}}} \log p_\sigma(\widetilde{\boldsymbol{x}})$. One particularly useful objective is denoising score matching, which avoids explicit evaluation of the perturbed density.

```{prf:theorem} Optimal target in denoising score matching
:label: thm-dsm-target

Consider the objective
:::{math}
\mathcal{L}_{DSM}(\theta)
=
\mathbb{E}_{\boldsymbol{x}, \widetilde{\boldsymbol{x}} | \boldsymbol{x}}
\left[
    \left\|
        \boldsymbol{s}_\theta(\widetilde{\boldsymbol{x}}, \sigma)
        -
        \nabla_{\widetilde{\boldsymbol{x}}}
        \log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
    \right\|_2^2
\right],
:::
where $q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x}) = \mathcal{N}(\widetilde{\boldsymbol{x}}; \boldsymbol{x}, \sigma^2 \boldsymbol{I})$. The minimizer of this objective is
:::{math}
\boldsymbol{s}_\theta^\star(\widetilde{\boldsymbol{x}}, \sigma)
=
\nabla_{\widetilde{\boldsymbol{x}}}\log p_\sigma(\widetilde{\boldsymbol{x}}).
:::
```

```{prf:proof}
For any fixed noisy observation $\widetilde{\boldsymbol{x}}$, the conditional expectation identity for squared error implies that the optimal predictor is
:::{math}
\mathbb{E}
\left[
    \nabla_{\widetilde{\boldsymbol{x}}}
    \log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
    \,|\,
    \widetilde{\boldsymbol{x}}
\right].
:::
Since
:::{math}
p_\sigma(\widetilde{\boldsymbol{x}})
=
\int q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x}) p_{gt}(\boldsymbol{x})\, d\boldsymbol{x},
:::
differentiating under the integral sign gives
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}} p_\sigma(\widetilde{\boldsymbol{x}})
=
\int
\nabla_{\widetilde{\boldsymbol{x}}} q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
p_{gt}(\boldsymbol{x})\, d\boldsymbol{x}.
:::
Dividing by $p_\sigma(\widetilde{\boldsymbol{x}})$ yields
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}\log p_\sigma(\widetilde{\boldsymbol{x}})
=
\int
\nabla_{\widetilde{\boldsymbol{x}}}
\log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
\, p(\boldsymbol{x} | \widetilde{\boldsymbol{x}})\, d\boldsymbol{x},
:::
which is exactly the conditional expectation above. Hence the optimal network equals the score of the perturbed density.
```

Because the Gaussian conditional score is explicit,
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}
\log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
=
-\frac{\widetilde{\boldsymbol{x}} - \boldsymbol{x}}{\sigma^2},
:::
the objective becomes a tractable regression problem. This is the first major bridge between diffusion modeling and denoising: learning a score field can be turned into learning how corruption relates noisy samples back to clean ones.

This theorem is one of the cleanest examples in the course of a hard generative objective being transformed into an easy regression target. The model is asked to predict a quantity derived from a Gaussian conditional that we know analytically, yet the minimizer recovers the score of the unknown noisy data distribution. In spirit, this mirrors what happened in DDPMs when the variational objective collapsed into noise prediction.

A useful teaching point is that the network is never asked to estimate the score of the empirical data distribution directly at zero noise. Instead, it is trained across a continuum or grid of noise scales where the distributions are smoother. The difficult object is therefore approached through a family of easier auxiliary problems. This pattern is one of the recurring design ideas of modern generative modeling.

## Forward SDEs and Their Marginals

In the continuous-time formulation, the noisy data process evolves according to an Itô SDE
:::{math}
d\boldsymbol{x}
=
\boldsymbol{f}(\boldsymbol{x}, t)\,dt
+
g(t)\,d\boldsymbol{w},
:::
where $\boldsymbol{w}$ is standard Brownian motion. The drift $\boldsymbol{f}$ and diffusion amplitude $g$ are chosen so that, as time increases, the distribution of $\boldsymbol{x}(t)$ becomes progressively simpler. One usually wants the terminal law at time $T$ to be close to a standard Gaussian that is easy to sample from.

Two particularly important families are the variance exploding and variance preserving SDEs. In the **VE-SDE**, the drift vanishes and noise strength grows over time:
:::{math}
d\boldsymbol{x} = \sqrt{\frac{d[\sigma^2(t)]}{dt}}\, d\boldsymbol{w}.
:::
Here the signal itself is not contracted by the drift, but the variance of the perturbation keeps increasing. In the **VP-SDE**, one uses
:::{math}
d\boldsymbol{x}
=
-\frac{1}{2}\beta(t)\boldsymbol{x}\,dt
+
\sqrt{\beta(t)}\, d\boldsymbol{w}.
:::
This process shrinks the signal while injecting noise, so the total variance remains controlled even though the initial information is gradually destroyed. For image generation, the VP family is especially important because it is the continuous-time analogue most closely connected to DDPMs.

The VP versus VE distinction is worth making more concrete because the acronyms can otherwise feel like catalog labels. In a VP process, the signal amplitude is explicitly damped by the drift while noise is added, so one can think of the original image as being gradually forgotten and replaced by a controlled-variance noisy state. In a VE process, the original signal is not contracted by a drift term; instead, the observation is overwhelmed by increasingly large additive noise. Both routes can end near a simple reference law, but they organize the path to that law differently.

This difference affects both theory and implementation. The natural parameterizations of time, the interpretation of the denoiser, and the numerical stiffness of the reverse dynamics can differ between VP and VE constructions. For the purposes of this course, the most important point is that DDPMs sit conceptually much closer to the VP family, whereas the broader score-based literature often discusses both VP and VE as parallel continuous-time noising choices.

## Reverse-Time SDE

The forward SDE explains how to corrupt data. Generation requires the reverse evolution that starts from a simple terminal distribution and moves toward the data distribution. The remarkable fact is that the **reverse-time dynamics** are again governed by an SDE, but with a drift corrected by the score of the time-marginal density.

```{prf:theorem} Reverse-time SDE
:label: thm-reverse-sde

Let the forward process satisfy
:::{math}
d\boldsymbol{x}
=
\boldsymbol{f}(\boldsymbol{x}, t)\,dt
+
g(t)\,d\boldsymbol{w},
:::
and let $p_t$ denote the density of $\boldsymbol{x}(t)$. Then, under suitable regularity assumptions, the reverse-time process satisfies
:::{math}
d\boldsymbol{x}
=
\left[
    \boldsymbol{f}(\boldsymbol{x}, t)
    -
    g^2(t)\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})
\right]dt
+
g(t)\,d\overline{\boldsymbol{w}},
:::
where time now runs backward and $\overline{\boldsymbol{w}}$ is a reverse-time Brownian motion.
```

```{prf:proof}
We sketch the derivation at the level of densities and probability fluxes. The forward Itô SDE implies the Fokker-Planck equation
:::{math}
\partial_t p_t(\boldsymbol{x})
=
-\nabla_{\boldsymbol{x}} \cdot
\left(
    \boldsymbol{f}(\boldsymbol{x}, t)p_t(\boldsymbol{x})
\right)
+
\frac{1}{2} g^2(t)\Delta p_t(\boldsymbol{x}).
:::
Since
:::{math}
\Delta p_t
=
\nabla_{\boldsymbol{x}} \cdot
\left(
    p_t \nabla_{\boldsymbol{x}} \log p_t
\right),
:::
we can rewrite the diffusion term as
:::{math}
\frac{1}{2} g^2(t)\Delta p_t
=
-\nabla_{\boldsymbol{x}} \cdot
\left(
    -\frac{1}{2} g^2(t)
    p_t
    \nabla_{\boldsymbol{x}} \log p_t
\right).
:::
Therefore the forward density evolution takes the continuity-equation form
:::{math}
\partial_t p_t(\boldsymbol{x})
=
-\nabla_{\boldsymbol{x}} \cdot
\left(
    p_t(\boldsymbol{x})
    \left[
        \boldsymbol{f}(\boldsymbol{x}, t)
        -
        \frac{1}{2} g^2(t)
        \nabla_{\boldsymbol{x}} \log p_t(\boldsymbol{x})
    \right]
\right).
:::

Now reverse time by setting $s = T-t$ and requiring the reversed process to traverse the same family of marginals in the opposite order. The sign of the probability flux must therefore flip. If the reverse process has drift $\widetilde{\boldsymbol{f}}$, its own Fokker-Planck equation contributes not only the transport term $-\nabla \cdot (\widetilde{\boldsymbol{f}} p_t)$ but also a new diffusion term $+\frac{1}{2} g^2 \Delta p_t$. Matching the reversed flux exactly thus requires subtracting another $\frac{1}{2} g^2 \nabla \log p_t$ from the drift. Combining the two halves gives
:::{math}
\widetilde{\boldsymbol{f}}(\boldsymbol{x}, t)
=
\boldsymbol{f}(\boldsymbol{x}, t)
-
g^2(t)\nabla_{\boldsymbol{x}} \log p_t(\boldsymbol{x}).
:::
Hence the reverse dynamics satisfy
:::{math}
d\boldsymbol{x}
=
\left[
    \boldsymbol{f}(\boldsymbol{x}, t)
    -
    g^2(t)\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})
\right]dt
+
g(t)\,d\overline{\boldsymbol{w}}.
:::
A fully rigorous proof requires time-reversal theory for diffusions, but this calculation captures the central mechanism: reversing a noising diffusion demands a score correction that steers mass back toward higher-density regions of the current marginal.
```

This theorem contains the conceptual heart of score-based diffusion. The reverse drift is not arbitrary. It equals the forward drift corrected by a term that points toward regions of higher density in the current marginal distribution. In other words, the score tells the reverse dynamics how to denoise. The only missing ingredient is that the true score is unknown, so it must be approximated by a neural network $\boldsymbol{s}_\theta(\boldsymbol{x}, t)$ trained from noisy data.

This is the place where the whole diffusion story condenses into one sentence: if you know the score of the time-marginal density at every noise level, then you know how to run the generative process backward. The unknown object in the reverse dynamics is therefore not an arbitrary conditional law but one geometrically meaningful vector field. This is why score estimation is not an auxiliary idea around diffusion; it is the central object that makes reverse sampling possible.

From the viewpoint of scientific understanding, this theorem is also satisfying because it separates modeling from simulation. Modeling means estimating the score field. Simulation means numerically integrating either the reverse SDE or a related deterministic ODE once that field is available. The same trained network can therefore support multiple samplers, which is one reason the diffusion literature contains so much work on fast and improved generation methods.

## Tweedie's Formula

Tweedie's formula makes the denoising interpretation even more explicit. It states that for Gaussian corruption, the posterior mean of the clean signal can be written directly in terms of the score of the noisy distribution.

```{prf:theorem} Tweedie's formula
:label: thm-tweedie

Let
:::{math}
\widetilde{\boldsymbol{x}} = \boldsymbol{x} + \sigma \boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}),
:::
and let $p_\sigma(\widetilde{\boldsymbol{x}})$ denote the density of the noisy observation. Then
:::{math}
\mathbb{E}[\boldsymbol{x} | \widetilde{\boldsymbol{x}}]
=
\widetilde{\boldsymbol{x}}
+
\sigma^2
\nabla_{\widetilde{\boldsymbol{x}}}
\log p_\sigma(\widetilde{\boldsymbol{x}}).
:::
```

```{prf:proof}
From the Gaussian conditional density,
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}
\log q_\sigma(\widetilde{\boldsymbol{x}} | \boldsymbol{x})
=
\frac{\boldsymbol{x} - \widetilde{\boldsymbol{x}}}{\sigma^2}.
:::
Taking the conditional expectation with respect to $\boldsymbol{x} | \widetilde{\boldsymbol{x}}$ and using the identity proved in the denoising score matching theorem gives
:::{math}
\nabla_{\widetilde{\boldsymbol{x}}}
\log p_\sigma(\widetilde{\boldsymbol{x}})
=
\frac{\mathbb{E}[\boldsymbol{x} | \widetilde{\boldsymbol{x}}] - \widetilde{\boldsymbol{x}}}{\sigma^2}.
:::
Rearranging yields the formula.
```

Tweedie's formula shows that a score estimator is implicitly a denoiser. Once the score of the noisy distribution is known, one can recover the posterior mean of the clean sample. This is one reason diffusion, score matching, and denoising are so tightly intertwined in the literature. They are not separate intuitions accidentally leading to similar algorithms; they are different views of the same mathematical identity.

This identity is pedagogically powerful because it translates an abstract differential quantity into an estimator of a familiar object. A student may wonder what the gradient of a log-density really buys us. Tweedie's formula answers: it tells us how to correct a noisy observation toward the posterior mean of the clean signal. In other words, the score is not just a geometric arrow field; it is directly actionable for denoising.

One should also notice the continuity with the discrete diffusion chapter. There, the network predicted noise and from that prediction one could reconstruct an estimate of $\boldsymbol{x}_0$. Here, the score plays the analogous role in continuous time. The mathematical packages differ, but the operational meaning is the same: infer how the current noisy sample should move toward cleaner structure.

## Probability Flow ODE

The reverse-time SDE is stochastic, but remarkably there exists a deterministic ODE with exactly the same time marginals. This is the **probability flow ODE**. It provides a bridge from stochastic diffusion models to deterministic transport models.

```{prf:theorem} Probability flow ODE
:label: thm-probability-flow

Consider the forward SDE
:::{math}
d\boldsymbol{x}
=
\boldsymbol{f}(\boldsymbol{x}, t)\,dt
+
g(t)\,d\boldsymbol{w}.
:::
The deterministic ODE
:::{math}
\frac{d\boldsymbol{x}}{dt}
=
\boldsymbol{f}(\boldsymbol{x}, t)
-
\frac{1}{2}g^2(t)\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})
:::
has the same marginal densities $p_t$ as the forward SDE when evolved in the same time direction.
```

```{prf:proof}
The proof follows by comparing the partial differential equations satisfied by the densities. The forward SDE induces a Fokker-Planck equation containing both drift and diffusion terms. One can rewrite the diffusion contribution as a divergence involving the score of the density. The ODE above is chosen so that its continuity equation produces exactly the same density evolution. The factor $\frac{1}{2}$ appears because the stochastic diffusion term contributes a second-order operator whose divergence form corresponds to half the square diffusion strength multiplied by the score field.
```

This theorem is extremely important conceptually. It says that the same score network can support both stochastic sampling, through the reverse SDE, and deterministic sampling, through the probability flow ODE. In practice, the ODE viewpoint also enables likelihood-related calculations and creates a clear conceptual link with continuous normalizing flows and, later in the course, with flow matching.

For many readers, this is the most surprising theorem in the chapter. One may expect that if the forward corruption used stochastic noise, then the reverse generative process must also be stochastic. The probability flow ODE says otherwise: there exists a deterministic evolution that shares the same one-time marginals as the stochastic reverse diffusion. This does not mean that individual trajectories match. It means that the distribution at each time is the same.

This observation is one of the clearest conceptual bridges between diffusion and later flow matching ideas. A score-based diffusion model can be viewed not only as a stochastic denoiser but also as a learned transport system. Once students understand this, the transition to deterministic transport methods becomes far smoother.

## VP-SDE, VE-SDE, and the Relation with DDPMs

We can now position the major continuous-time families more clearly. In the VP-SDE,
:::{math}
d\boldsymbol{x}
=
-\frac{1}{2}\beta(t)\boldsymbol{x}\,dt
+
\sqrt{\beta(t)}\, d\boldsymbol{w},
:::
the mean signal decays exponentially while noise is injected continuously. This is the continuous analogue of the discrete Gaussian noising process used in DDPMs, and for this reason it is the main focus of these notes. In the VE-SDE, the drift is zero and the noise standard deviation grows directly with time. Both constructions produce families of perturbed densities whose scores can be learned, but they differ in numerical behavior and in the geometry of the perturbation path.

From the discrete viewpoint, DDPM training often predicts the noise $\boldsymbol{\varepsilon}$. From the continuous-time viewpoint, the more intrinsic object is the score $\nabla_{\boldsymbol{x}}\log p_t(\boldsymbol{x})$. The two parameterizations are equivalent up to analytic transformations determined by the perturbation law. This is why discrete diffusion, denoising score matching, and continuous-time score-based modeling form one coherent family rather than disconnected algorithms.

This is also the right place to mention latent diffusion from the continuous-time viewpoint. Once score or noise prediction is understood as learning a reverse evolution in some representation space, there is no requirement that this space be the raw pixel grid. If an autoencoder provides a compressed latent representation in which semantics are preserved reasonably well, then the same VP- or VE-style reasoning can be applied there. The practical gain is enormous because the differential evolution is solved in a lower-dimensional and often smoother space. The conceptual cost is that one must now trust the latent representation as part of the generative model.

## Final Perspective for the Course

At this point the continuous-time picture is in place. The forward process spreads data toward a simple reference distribution. The score network learns how density changes across that perturbation path. The reverse-time SDE converts this learned score into a stochastic sampler, while the probability flow ODE converts the same score into a deterministic sampler with identical marginals. Tweedie's formula explains why score estimation and denoising are two sides of the same coin.

From the broader perspective of the course, diffusion is important not only because it is a successful model family, but because it teaches a very modern lesson about generative modeling. One can obtain powerful generators by designing a path between data and noise, learning the local geometry of that path, and then choosing an efficient reverse solver. This path-centric view will reappear in flow matching, where the stochastic path is replaced by a directly parameterized transport field.

The next notebook will turn this theory into code. There we will implement sinusoidal time embeddings, a compact U-Net style denoiser, the noisy sample construction, and a basic sampling loop. The main references for this chapter are {cite}`song2021scorebased`, {cite}`song2020denoising`, {cite}`song2021maximum`, and {cite}`song2020ddim`.

```{figure} ../assets/images/sinusoidal_embedding.png
:width: 72%
:align: center

Time-conditioning is a central ingredient in practical diffusion models, since the network must know which noise level or time point it is denoising.
```